# Pronóstico 6→6 especializado por ingreso, clúster 2

[← Metodología común](metodologia-pronostico-6x6.ipynb) ·
[← Resultados del modelo conjunto](cluster2.ipynb)

Este notebook es un experimento independiente. El modelo seleccionado en
`cluster2.ipynb` es el benchmark conjunto, donde un solo estimador genera los 30
objetivos. Aquí se ajusta una suite por cada uno de los cinco ingresos y se construye
una solución híbrida seleccionada exclusivamente con validación histórica.


In [1]:
# Configuración editable
from utils.forecasting_cache import ForecastCacheConfig

CLUSTER_COLUMN = "cluster2"
TRAINING_START = "2014-01-01"
PREFERRED_MUNICIPALITY = "Santiago"
TOTAL_WAPE_GUARDRAIL = 0.01

MODEL_CACHE = ForecastCacheConfig(
    directory="model_cache",
    enabled=True,
    refresh=False,
)

# Opcional: puede asignarse aquí un resultado ya calculado por
# run_cluster_forecasting_workflow con la misma configuración y los mismos cortes.
JOINT_WORKFLOW_RESULT = None


## Punto de partida del experimento especializado

El experimento reutiliza las comunas y el inicio de entrenamiento de `cluster2`, junto
con el contrato, los cortes y la cobertura definidos en la
[metodología común](metodologia-pronostico-6x6.ipynb). Su única diferencia comienza en
la arquitectura de salida y en la regla que combina especialistas.


In [2]:
import warnings

import pandas as pd
from IPython.display import HTML, display

from utils.data import data_loader
from utils.forecasting import FORECAST_INCOME_GROUPS
from utils.forecasting_cache import summarize_cache_report
from utils.forecasting_workflow import resolve_cluster_municipalities
from utils.notebook_display import collapsible_stdout, display_collapsible
from utils.plots import (
    plot_cluster_municipality_improvement,
    plot_hybrid_combination_frontier,
    plot_joint_hybrid_improvement_matrix,
    plot_specialized_family_comparison,
)
from utils.separated_forecasting import (
    AVERAGE_INCOME_GROUP_LABEL,
    HYBRID_ARCHITECTURE,
    JOINT_ARCHITECTURE,
    SATISFACTORY_IMPROVEMENT_COLUMNS,
    TEST_PERIOD_LABEL,
    build_joint_hybrid_improvement_summary,
    build_separated_municipality_forecast_view,
    run_separated_income_forecasting_workflow,
)

warnings.filterwarnings(
    "ignore",
    message=(
        r"`sklearn\.utils\.parallel\.delayed` should be used with "
        r"`sklearn\.utils\.parallel\.Parallel`.*"
    ),
    category=UserWarning,
    module=r"sklearn\.utils\.parallel",
)

clusters = pd.read_csv("clusters.csv")
municipalities = resolve_cluster_municipalities(clusters, CLUSTER_COLUMN)
with collapsible_stdout("Ver registro de carga de datos"):
    presupuesto_cluster = data_loader(municipalities=municipalities)

display_collapsible(
    "Ver configuración efectiva",
    pd.DataFrame(
        {
            "cluster": [CLUSTER_COLUMN],
            "comunas": [len(municipalities)],
            "inicio de entrenamiento": [pd.Timestamp(TRAINING_START)],
            "comuna para detalle": [PREFERRED_MUNICIPALITY],
            "guardrail WAPE total": [TOTAL_WAPE_GUARDRAIL],
            "caché habilitado": [MODEL_CACHE.enabled],
            "refrescar caché": [MODEL_CACHE.refresh],
            "directorio caché": [str(MODEL_CACHE.directory)],
        }
    )
)
municipality_label = "comuna" if len(municipalities) == 1 else "comunas"
municipality_items = "".join(
    f"<li>{municipality}</li>" for municipality in municipalities
)
display_collapsible(
    f"Ver {len(municipalities)} {municipality_label} del clúster",
    HTML(
        f'<ul class="cluster-members__list">{municipality_items}</ul>'
    )
)


,cluster,comunas,inicio de entrenamiento,comuna para detalle,guardrail WAPE total,caché habilitado,refrescar caché,directorio caché
0,cluster2,25,2014-01-01,Santiago,0.01,True,False,model_cache


## Arquitectura conjunta frente a arquitecturas especializadas

Todos los modelos reciben las mismas entradas del contrato común. La diferencia está
únicamente en la salida.

| Arquitectura | Modelos entrenados | Salida directa | Total disponible |
|---|---:|---:|---|
| Conjunta | 1 | 30 objetivos, 6 meses × 5 ingresos | suma de los cinco ingresos |
| Separada | 5 | 6 objetivos por modelo, uno por ingreso | suma exacta de las cinco salidas |
| Híbrida | hasta 5 especializados más el conjunto | elige una fuente por ingreso | suma exacta de las fuentes elegidas |

> **Costo esperado:** el tuning especializado requiere aproximadamente cinco veces
> más ajustes que el notebook conjunto. Si se entrega `JOINT_WORKFLOW_RESULT`, la
> reutilización en memoria del benchmark tiene prioridad sobre el caché en disco.


## Regla de selección congelada

Para cada ingreso se ajustan Ridge, ExtraTrees y CatBoost con 2018–2022. Luego esas
tres familias y Persistencia compiten en tres cortes de validación cuyos objetivos
abarcan enero de 2024 a junio de 2025.

Con los cinco ganadores especializados se evalúan las 32 combinaciones posibles entre
fuente conjunta y especializada. Para cada combinación se calcula la mejora relativa
de WAPE frente al modelo conjunto en `Transferencias corrientes`, `Transferencias de
capital` y `Otros ingresos`, y se promedian esas tres mejoras con igual peso.

Una combinación híbrida es elegible solo si ese promedio es estrictamente mayor que
5 % y su WAPE macro de `Total disponible` no empeora en más de 0,01 —un punto
porcentual— respecto del conjunto. Entre las alternativas elegibles se elige la de
mayor mejora prioritaria; los desempates favorecen menor WAPE total, menor WAPE
promedio de los cinco ingresos, menos modelos especializados y un orden determinista.
Si ninguna cumple ambos criterios, se selecciona `00000`, el modelo completamente
conjunto. El mapa resultante se congela antes de calcular cualquier métrica de 2026.

Las definiciones de WAPE macro, WAPE micro, MAE y sesgo se mantienen exactamente como
en la [metodología común](metodologia-pronostico-6x6.ipynb).


In [3]:
with collapsible_stdout("Ver registro de entrenamiento especializado"):
    workflow = run_separated_income_forecasting_workflow(
        presupuesto_cluster,
        municipalities,
        cluster_label=CLUSTER_COLUMN,
        training_start=TRAINING_START,
        joint_result=JOINT_WORKFLOW_RESULT,
        total_guardrail=TOTAL_WAPE_GUARDRAIL,
        progress=True,
        cache_config=MODEL_CACHE,
    )

display_collapsible("Ver configuración del workflow", workflow.configuration)
display_collapsible("Ver ventanas de entrenamiento", workflow.training_windows)
display_collapsible(
    "Ver resumen del caché",
    summarize_cache_report(workflow.cache_report),
)

cache_by_stage = (
    workflow.cache_report.groupby(
        ["stage", "artifact_type", "status"], sort=False, dropna=False
    )
    .agg(artefactos=("key", "size"), gb=("size_bytes", lambda x: x.sum() / 1024**3))
    .reset_index()
)
display_collapsible("Ver artefactos de caché por etapa", cache_by_stage)


,cluster,comunas,inicio_entrenamiento_solicitado,primer_corte_tuning,ultimo_corte_tuning,primer_corte_validacion,ultimo_corte_validacion,fin_entrenamiento_final,entrada_final,test_congelado,ventanas_entrenamiento_final,variables_mensuales_por_modelo,objetivos_por_modelo_especializado,suites_especializadas,combinaciones_hibridas,guardrail_total
0,cluster2,25,2014-01-01,2018-06-01,2022-12-01,2023-06-01,2024-06-01,2025-06-01,julio-diciembre 2025,enero-junio 2026,3175,30,6,5,32,0.01


,Nombre Municipio,primera_entrada,ultimo_objetivo,ventanas
0,Cerrillos,2014-01-01,2025-06-01,127
1,Cerro Navia,2014-01-01,2025-06-01,127
2,Curacaví,2014-01-01,2025-06-01,127
3,El Bosque,2014-01-01,2025-06-01,127
4,El Monte,2014-01-01,2025-06-01,127
5,Estación Central,2014-01-01,2025-06-01,127
6,Independencia,2014-01-01,2025-06-01,127
7,Isla de Maipo,2014-01-01,2025-06-01,127
8,La Cisterna,2014-01-01,2025-06-01,127
9,La Granja,2014-01-01,2025-06-01,127


,hits,misses,modelos_cargados,evaluaciones_cargadas,reutilizaciones_memoria,artefactos_escritos,artefactos_corruptos_aislados,gb_cargados,gb_escritos,gb_artefactos_utilizados
0,687,0,8,679,0,0,0,0.573365,0.0,0.573365


,stage,artifact_type,status,artefactos,gb
0,joint_tuning,evaluation,hit,110,0.001555
1,joint_validation,evaluation,hit,3,0.000088
2,specialized_tuning:Transferencias corrientes,evaluation,hit,110,0.000375
3,specialized_validation:Transferencias corrientes,evaluation,hit,3,0.000018
4,specialized_tuning:Transferencias de capital,evaluation,hit,110,0.000355
5,specialized_validation:Transferencias de capital,evaluation,hit,3,0.000017
6,specialized_tuning:Otros ingresos,evaluation,hit,110,0.000395
7,specialized_validation:Otros ingresos,evaluation,hit,3,0.000019
8,specialized_tuning:IPP,evaluation,hit,110,0.000394
9,specialized_validation:IPP,evaluation,hit,3,0.000019


## Del modelo de `cluster2.ipynb` a los especialistas

El benchmark conserva el ganador conjunto seleccionado por WAPE macro de `Total
disponible` en `cluster2.ipynb`. La tabla identifica los cinco ganadores especializados
y el mapa de calor muestra cuánto se separa cada familia del mejor modelo de su
ingreso. Ninguna de estas decisiones usa 2026.


In [4]:
joint_ranking = workflow.joint_validation_ranking.copy()
joint_ranking["WAPE macro (%)"] = (100 * joint_ranking["wape_macro"]).round(2)
joint_ranking["WAPE micro (%)"] = (100 * joint_ranking["wape_micro"]).round(2)
display_collapsible(
    "Ver ranking del modelo conjunto",
    joint_ranking[
        [
            "ranking",
            "seleccion_validacion",
            "modelo",
            "WAPE macro (%)",
            "WAPE micro (%)",
            "mae_mm_clp",
            "sesgo_micro_mm_clp",
        ]
    ]
)

comparison_summary = build_joint_hybrid_improvement_summary(workflow)
specialized_winners = workflow.specialized_winners.copy()
specialized_winners["WAPE macro validación (%)"] = (
    100 * specialized_winners["wape_macro_validacion"]
).round(2)
specialized_winners["WAPE micro validación (%)"] = (
    100 * specialized_winners["wape_micro_validacion"]
).round(2)
display_collapsible(
    "Ver modelos especializados seleccionados",
    specialized_winners[
        [
            "grupo_ingreso",
            "modelo_especializado",
            "candidate_id",
            "parametros",
            "WAPE macro validación (%)",
        ]
    ]
)

family_figure = plot_specialized_family_comparison(
    workflow.joint_validation_summary,
    workflow.specialized_validation_summary,
    workflow.specialized_winners,
    joint_model_name=workflow.joint_selected_model_name,
)
display(
    HTML(
        family_figure.to_html(
            full_html=False,
            include_plotlyjs="cdn",
            config={"responsive": True, "displaylogo": False},
        )
    )
)


,ranking,seleccion_validacion,modelo,WAPE macro (%),WAPE micro (%),mae_mm_clp,sesgo_micro_mm_clp
0,1,True,ExtraTrees global,19.45,18.91,647.506706,-362.274264
1,2,False,CatBoost global,21.56,20.37,697.692237,-293.584241
2,3,False,Ridge global,22.45,20.64,706.927985,-149.227974
3,4,False,Persistencia,44.05,43.36,1484.804413,-72.060067


,grupo_ingreso,modelo_especializado,candidate_id,parametros,WAPE macro validación (%)
0,Transferencias corrientes,CatBoost global,catboost_depth_6_iterations_300,"{'iterations': 300, 'depth': 6, 'learning_rate...",102.11
1,Transferencias de capital,CatBoost global,catboost_depth_6_iterations_300,"{'iterations': 300, 'depth': 6, 'learning_rate...",163.03
2,Otros ingresos,CatBoost global,catboost_depth_6_iterations_300,"{'iterations': 300, 'depth': 6, 'learning_rate...",63.57
3,IPP,ExtraTrees global,extra_trees_depth_none_leaf_3,"{'n_estimators': 400, 'max_depth': None, 'min_...",18.63
4,FCM,ExtraTrees global,extra_trees_depth_none_leaf_3,"{'n_estimators': 400, 'max_depth': None, 'min_...",20.43


## Frontera de las 32 combinaciones y mapa congelado

El eje vertical muestra la mejora relativa promedio en los tres ingresos prioritarios:
más arriba es mejor. El eje horizontal muestra el WAPE macro de `Total disponible`:
más a la izquierda es mejor. La línea horizontal marca el requisito estricto de 5 % y
la vertical, el guardrail del total; solo las combinaciones ubicadas en la zona verde
son elegibles. El color indica cuántos ingresos usan un especialista y la línea verde
une las combinaciones no dominadas.

Las decisiones centrales se etiquetan directamente: `00000` es el modelo conjunto y
la combinación congelada se destaca como seleccionada. En cada código, `0` significa
fuente conjunta y `1`, fuente especializada, siguiendo el orden canónico de los cinco
ingresos.

La tabla conserva las cinco mejores combinaciones y `00000` como referencia auditable
del modelo conjunto.


In [5]:
combination_ranking = workflow.combination_ranking.copy()
combination_ranking["WAPE promedio ingresos (%)"] = (
    100 * combination_ranking["wape_macro_promedio_ingresos"]
).round(2)
combination_ranking["WAPE total (%)"] = (
    100 * combination_ranking["wape_macro_total"]
).round(2)
combination_ranking["Límite total (%)"] = (
    100 * combination_ranking["limite_wape_total"]
).round(2)
combination_ranking["Mejora prioritaria promedio (%)"] = (
    100 * combination_ranking["mejora_relativa_promedio_prioritaria"]
).round(2)
priority_improvement_labels = {
    column: f"Mejora {group} (%)"
    for group, column in SATISFACTORY_IMPROVEMENT_COLUMNS
}
for source_column, display_column in priority_improvement_labels.items():
    combination_ranking[display_column] = (
        100 * combination_ranking[source_column]
    ).round(2)
combination_audit = pd.concat(
    [
        combination_ranking.head(5),
        combination_ranking.loc[combination_ranking["codigo"].eq("00000")],
    ],
    ignore_index=True,
).drop_duplicates("codigo")
display_collapsible(
    "Ver ranking de combinaciones híbridas",
    combination_audit[
        [
            "ranking",
            "seleccionada",
            "codigo",
            "ingresos_especializados",
            "elegible",
            "criterio_mejora_prioritaria_cumplido",
            "guardrail_cumplido",
            "Mejora prioritaria promedio (%)",
            *priority_improvement_labels.values(),
            "WAPE promedio ingresos (%)",
            "WAPE total (%)",
            "Límite total (%)",
        ]
    ]
)

source_map = pd.DataFrame(
    [
        {
            "grupo_ingreso": group,
            "fuente_final": workflow.selected_source_map[group],
        }
        for group in FORECAST_INCOME_GROUPS
    ]
)
display_collapsible("Ver mapa de fuentes seleccionado", source_map)

acceptance_display = workflow.acceptance.copy()
for source_column, display_column in priority_improvement_labels.items():
    acceptance_display[display_column] = (
        100 * acceptance_display[source_column]
    ).round(2)
acceptance_display["Mejora prioritaria promedio (%)"] = (
    100 * acceptance_display["mejora_relativa_promedio_prioritaria"]
).round(2)
acceptance_display["Umbral estricto (%)"] = (
    100 * acceptance_display["umbral_mejora_relativa_prioritaria"]
).round(2)
display_collapsible(
    "Ver criterio de mejora satisfactoria",
    acceptance_display[
        [
            "codigo_seleccionado",
            *priority_improvement_labels.values(),
            "Mejora prioritaria promedio (%)",
            "Umbral estricto (%)",
            "criterio_mejora_prioritaria_cumplido",
            "guardrail_cumplido",
            "combinaciones_hibridas_elegibles",
            "mejora_satisfactoria",
            "motivo_seleccion",
        ]
    ],
)

combination_figure = plot_hybrid_combination_frontier(
    workflow.combination_ranking
)
display(
    HTML(
        combination_figure.to_html(
            full_html=False,
            include_plotlyjs="cdn",
            config={"responsive": True, "displaylogo": False},
        )
    )
)


,ranking,seleccionada,codigo,ingresos_especializados,elegible,criterio_mejora_prioritaria_cumplido,guardrail_cumplido,Mejora prioritaria promedio (%),Mejora Transferencias corrientes (%),Mejora Transferencias de capital (%),Mejora Otros ingresos (%),WAPE promedio ingresos (%),WAPE total (%),Límite total (%)
0,1,True,11111,5,True,True,True,15.63,15.22,21.62,10.05,73.55,16.99,20.45
1,2,False,11101,4,True,True,True,15.63,15.22,21.62,10.05,73.88,17.59,20.45
2,3,False,11110,4,True,True,True,15.63,15.22,21.62,10.05,73.76,18.34,20.45
3,4,False,11100,3,True,True,True,15.63,15.22,21.62,10.05,74.08,19.34,20.45
4,5,False,11011,4,True,True,True,12.28,15.22,21.62,0.00,74.97,16.62,20.45
5,32,False,00000,0,False,False,True,0.00,0.00,0.00,0.00,88.16,19.45,20.45


,grupo_ingreso,fuente_final
0,Transferencias corrientes,especializado
1,Transferencias de capital,especializado
2,Otros ingresos,especializado
3,IPP,especializado
4,FCM,especializado


,codigo_seleccionado,Mejora Transferencias corrientes (%),Mejora Transferencias de capital (%),Mejora Otros ingresos (%),Mejora prioritaria promedio (%),Umbral estricto (%),criterio_mejora_prioritaria_cumplido,guardrail_cumplido,combinaciones_hibridas_elegibles,mejora_satisfactoria,motivo_seleccion
0,11111,15.22,21.62,10.05,15.63,5.0,True,True,24,True,híbrida: mayor mejora relativa promedio en ing...


## Cuánto mejora frente a `cluster2.ipynb`

La comparación usa como línea base el ganador conjunto de `cluster2.ipynb`. La
diferencia en puntos porcentuales es `WAPE híbrido - WAPE conjunto`: un valor negativo
indica menor error. La mejora relativa usa al conjunto como denominador. La selección
se decide con tres cortes históricos cuyos objetivos abarcan enero de 2024 a junio de
2025. Después, julio-diciembre de 2025 se usa como entrada final y enero-junio de 2026
queda reservado como test congelado. El test 2026 solo mide cuánto se sostuvo fuera de
muestra y no puede cambiar la arquitectura seleccionada.


In [6]:
executive_comparison = comparison_summary.loc[
    comparison_summary["grupo_ingreso"].isin(
        ["Total disponible", AVERAGE_INCOME_GROUP_LABEL]
    )
].copy()
executive_comparison["WAPE conjunto (%)"] = (
    100 * executive_comparison["wape_conjunto"]
).round(2)
executive_comparison["WAPE híbrido (%)"] = (
    100 * executive_comparison["wape_hibrido"]
).round(2)
executive_comparison["Mejora relativa (%)"] = (
    100 * executive_comparison["mejora_relativa"]
).round(2)
display_collapsible(
    "Ver comparación ejecutiva conjunto versus híbrido",
    executive_comparison[
        [
            "periodo",
            "grupo_ingreso",
            "modelo_conjunto",
            "WAPE conjunto (%)",
            "WAPE híbrido (%)",
            "diferencia_wape_pp",
            "Mejora relativa (%)",
        ]
    ]
)

improvement_figure = plot_joint_hybrid_improvement_matrix(
    comparison_summary,
    joint_model_name=workflow.joint_selected_model_name,
)
display(
    HTML(
        improvement_figure.to_html(
            full_html=False,
            include_plotlyjs="cdn",
            config={"responsive": True, "displaylogo": False},
        )
    )
)


,periodo,grupo_ingreso,modelo_conjunto,WAPE conjunto (%),WAPE híbrido (%),diferencia_wape_pp,Mejora relativa (%)
0,Validación histórica · objetivos ene-2024 a ju...,Total disponible,ExtraTrees global,19.45,16.99,-2.463009,12.66
1,Validación histórica · objetivos ene-2024 a ju...,Promedio de los cinco ingresos,ExtraTrees global,88.16,73.55,-14.610396,16.57
7,Test congelado · objetivos ene-jun 2026,Total disponible,ExtraTrees global,16.35,16.02,-0.327752,2.00
8,Test congelado · objetivos ene-jun 2026,Promedio de los cinco ingresos,ExtraTrees global,114.98,105.83,-9.152256,7.96


## Lectura del test congelado de 2026

Solo después de fijar el mapa anterior se comparan las cuatro alternativas sobre
enero–junio de 2026. La máscara de meses observados es la misma del benchmark, de modo
que la comparación conjunta, separada e híbrida usa idéntica cobertura efectiva.


In [7]:
test_comparison = comparison_summary.loc[
    comparison_summary["periodo"].eq(TEST_PERIOD_LABEL)
    & ~comparison_summary["grupo_ingreso"].eq(AVERAGE_INCOME_GROUP_LABEL)
].copy()
test_comparison["WAPE conjunto (%)"] = (
    100 * test_comparison["wape_conjunto"]
).round(2)
test_comparison["WAPE híbrido (%)"] = (
    100 * test_comparison["wape_hibrido"]
).round(2)
test_comparison["Mejora relativa (%)"] = (
    100 * test_comparison["mejora_relativa"]
).round(2)
display_collapsible(
    "Ver comparación del test congelado de 2026",
    test_comparison[
        [
            "grupo_ingreso",
            "WAPE conjunto (%)",
            "WAPE híbrido (%)",
            "diferencia_wape_pp",
            "Mejora relativa (%)",
            "mejora",
        ]
    ]
)


,grupo_ingreso,WAPE conjunto (%),WAPE híbrido (%),diferencia_wape_pp,Mejora relativa (%),mejora
7,Total disponible,16.35,16.02,-0.327752,2.00,True
9,Transferencias corrientes,126.20,117.97,-8.233062,6.52,True
10,Transferencias de capital,298.03,298.00,-0.025699,0.01,True
11,Otros ingresos,113.66,76.68,-36.980871,32.54,True
12,IPP,17.40,17.29,-0.103785,0.60,True
13,FCM,19.61,19.19,-0.417862,2.13,True


## Cobertura efectiva del test

La cobertura se calcula una vez por comuna sobre `Total disponible` para la solución
híbrida. Una comuna con menos de seis meses evaluados tiene meses finales no
reportados que no participan en WAPE, MAE ni sesgo.


In [8]:
coverage_details = workflow.test_details.loc[
    workflow.test_details["modelo"].eq(HYBRID_ARCHITECTURE)
    & workflow.test_details["grupo_ingreso"].eq("Total disponible")
].copy()
coverage = (
    coverage_details.groupby("Nombre Municipio", sort=True, observed=True)
    .agg(
        meses_evaluados=("observado_mm_clp", lambda values: values.notna().sum()),
        meses_no_reportados=(
            "fecha",
            lambda dates: [
                date.strftime("%Y-%m")
                for date in dates[
                    coverage_details.loc[dates.index, "observado_mm_clp"].isna()
                ]
            ],
        ),
    )
    .reset_index()
)
coverage["cobertura_completa"] = coverage["meses_evaluados"].eq(6)
display_collapsible(
    "Ver comunas con cobertura incompleta",
    coverage.loc[~coverage["cobertura_completa"]],
)
display_collapsible(
    "Ver cobertura total evaluada",
    (
        f'Meses comuna evaluados: {int(coverage["meses_evaluados"].sum())} '
        f"de {6 * len(coverage)}"
    ),
)


,Nombre Municipio,meses_evaluados,meses_no_reportados,cobertura_completa
0,Cerrillos,5,[2026-06],False
1,Cerro Navia,5,[2026-06],False
4,El Monte,5,[2026-06],False
6,Independencia,5,[2026-06],False
14,Paine,4,"[2026-05, 2026-06]",False
18,Puente Alto,3,"[2026-04, 2026-05, 2026-06]",False
20,Recoleta,5,[2026-06],False


## Dónde mejora dentro del clúster y detalle configurable

Cambie `PREFERRED_MUNICIPALITY` en la celda inicial para inspeccionar otra comuna del
clúster. El panel izquierdo ordena las comunas por la reducción relativa del WAPE total
en el test de enero a junio de 2026. Una barra verde positiva significa que el híbrido
reduce el error frente al ExtraTrees conjunto; una barra naranja negativa significa
que lo aumenta. La etiqueta `n/6` informa cuántos meses tenían datos observados y
entraron realmente en la métrica, por lo que los resultados con 4/6 o 5/6 deben
interpretarse con más cautela.

El círculo identifica la comuna elegida y el panel derecho permite entender su barra:
compara mes a mes lo observado, Persistencia, el modelo conjunto de `cluster2.ipynb` y
el híbrido. El WAPE mostrado en la leyenda se calcula solo sobre los meses reportados.
Por eso el híbrido puede mejorar el resultado agregado del clúster y, al mismo tiempo,
ser peor para una comuna particular. `Separado completo` no se duplica cuando coincide
con el híbrido.


In [9]:
municipality_view = build_separated_municipality_forecast_view(
    workflow,
    PREFERRED_MUNICIPALITY,
)

municipality_metrics = municipality_view.metrics.loc[
    municipality_view.metrics["grupo_ingreso"].eq("Total disponible")
    & municipality_view.metrics["modelo"].isin(
        ["Persistencia", JOINT_ARCHITECTURE, HYBRID_ARCHITECTURE]
    )
].copy()
municipality_metrics["WAPE total (%)"] = (
    100 * municipality_metrics["wape"]
).round(2)
display_collapsible(
    f"Ver métricas de {municipality_view.municipality}",
    municipality_metrics[
        [
            "modelo",
            "WAPE total (%)",
            "mae_mm_clp",
            "sesgo_mm_clp",
            "meses_evaluados",
        ]
    ]
)

municipality_figure = plot_cluster_municipality_improvement(
    workflow.test_unit_metrics,
    municipality_view.actual,
    municipality_view.forecasts,
    municipality_view.metrics,
    municipality=municipality_view.municipality,
    joint_model_name=workflow.joint_selected_model_name,
)
display(
    HTML(
        municipality_figure.to_html(
            full_html=False,
            include_plotlyjs="cdn",
            config={"responsive": True, "displaylogo": False},
        )
    )
)


,modelo,WAPE total (%),mae_mm_clp,sesgo_mm_clp,meses_evaluados
3,Conjunto seleccionado,14.77,2161.219012,-1315.288180,6.0
9,Híbrido seleccionado,15.32,2241.032559,-1824.696190,6.0
15,Persistencia,44.61,6526.358976,511.221305,6.0


## Conclusión metodológica del experimento

La combinación híbrida se adopta solamente cuando la mejora relativa promedio de
`Transferencias corrientes`, `Transferencias de capital` y `Otros ingresos` supera
estrictamente el 5 % en validación histórica y, al mismo tiempo, se respeta el
guardrail de `Total disponible`. Si ninguna combinación cumple ambas condiciones, el
flujo vuelve al modelo conjunto `00000`.

El test congelado de enero a junio de 2026 se utiliza solo para comprobar cuánto se
sostuvo el desempeño fuera de muestra. Una mejora posterior aporta evidencia
diagnóstica, pero no puede modificar la combinación seleccionada con objetivos de
enero de 2024 a junio de 2025.
